# 13 — Reexperimentos de modelos de grafos (ABIDE I / CC200)

Notebook opcional para comparar arquiteturas e ablações de GCN/GAT usando os cinco folds agrupados por centro já definidos. Não refaz downloads, CPAC, ROIs, conectividade nem Fisher Z.

A matriz funcional completa (19.900 arestas) é a entrada de referência; variantes de máscara de arestas e pooling são ablações. O roster principal de modelos de grafos está no notebook 10 (GCN, GAT e BrainGNN).

**Protocolo:** configurações e épocas são selecionadas em splits internos agrupados por centro; o fold externo é usado somente para avaliação. Checkpoints e CSVs progressivos ficam em `Results/reexperimentos_13/`.


In [ ]:
# 1. Caminhos, dados e protocolo herdado
import os, sys, json, pickle, hashlib, time, warnings, copy, random
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
while not (ROOT/'src'/'config.py').exists():
    if ROOT.parent == ROOT: raise RuntimeError('Abra o notebook dentro do projeto TCC_TEA.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import PATHS, SEED
from src import avaliacao as av
OUT = PATHS['results']/'reexperimentos_13'; CK = OUT/'checkpoints'
OUT.mkdir(parents=True, exist_ok=True); CK.mkdir(parents=True, exist_ok=True)
DATASET = PATHS['processed']/'dataset_abide_processado.pkl'
MATRIX = PATHS['conectivity']/'connectivity_matrices_abide_validado.pkl'
d = av.carregar_dataset(DATASET)
folds = sorted(np.unique(d['fold_id']).tolist())
assert len(folds)==5, f'Esperados cinco folds existentes, achei {len(folds)}.'
av.resumo_folds(d)  # valida ausência de sobreposição de centros
# fingerprint: IDs, labels, centro, folds e conectividade; identifica exatamente os dados usados
h=hashlib.sha256()
for a in (d['X'], d['y'], d['ids'].astype('U'), d['sites'].astype('U'), d['fold_id']): h.update(np.ascontiguousarray(a).tobytes())
DATA_HASH=h.hexdigest()
with open(MATRIX,'rb') as f: mats_obj=pickle.load(f)
print('Dataset:', d['X'].shape, '| ROIs:', av.n_rois_de(d['X'].shape[1]), '| folds:', folds)
print('Participantes:',len(d['y']),'| centros:',len(np.unique(d['sites'])),'| dados SHA256:',DATA_HASH[:16])
print('Matriz validada lida:', type(mats_obj).__name__, '| arquivo:', MATRIX.name)
# usamos X do dataset processado (Fisher Z, mesma ordem triu); matriz validada é conferida abaixo


In [ ]:
# 2. Validação das matrizes existentes e alinhamento por ID
# O arquivo validado guarda matrizes Pearson R, enquanto dataset.X guarda Fisher Z.
# Logo, igualdade direta não é esperada: tanh(dataset.X) deve recuperar R.
ids_dataset=list(map(str,d["ids"]))
missing=[sid for sid in ids_dataset if sid not in mats_obj]
if missing:
    raise KeyError(f"{len(missing)} IDs do dataset não existem no PKL validado; exemplos: {missing[:5]}")
N=av.n_rois_de(d["X"].shape[1]); _i,_j=np.triu_indices(N,1)
MATS_VALID=np.stack([np.asarray(mats_obj[sid],dtype=np.float32) for sid in ids_dataset])
MATS_VALID[:,np.arange(N),np.arange(N)]=0.0
R_DATA=np.zeros_like(MATS_VALID)
R_DATA[:,_i,_j]=np.tanh(d["X"]); R_DATA[:,_j,_i]=np.tanh(d["X"])
err=float(np.max(np.abs(MATS_VALID-R_DATA)))
if not np.allclose(MATS_VALID,R_DATA,rtol=1e-4,atol=1e-5):
    raise ValueError(f"Matrizes validadas não correspondem ao dataset Fisher Z após alinhamento por ID (erro máximo={err:.3g}).")
R=MATS_VALID
phen = pd.read_csv(PATHS['raw']/'phenotypic_abide.csv') if (PATHS['raw']/'phenotypic_abide.csv').exists() else None
print(f"Matrizes Pearson alinhadas por ID e conferidas contra tanh(Fisher Z): n={len(MATS_VALID)}, erro máximo={err:.2g}")
print('Fenótipos brutos disponíveis:', phen is not None)


## Definição prévia da bateria

O modo padrão executa a bateria completa, o que pode ser demorado. `MODO_RAPIDO=True` reduz épocas/variantes apenas para checagem funcional e **não** deve ser usado como resultado de TCC. Cada experimento muda um componente por vez sempre que possível. A variante `principal` mantém todas as conexões funcionais. Variáveis fenotípicas só entram se forem identificadas e alinhadas por ID; caso contrário a comparação fica registrada como não disponível, sem fabricar informação. Atributos temporais/topológicos são opcionais e exigem os artefatos existentes.

In [ ]:
# 3. Configuração; hiperparâmetros escolhidos somente na validação interna
MODO_RAPIDO = False
SEMENTE_BASE = int(SEED)
EPOCHS = 4 if MODO_RAPIDO else 60
PATIENCE = 2 if MODO_RAPIDO else 9
BATCH = 24
# Ablações somente para GCN/GAT; BrainGNN fica no protocolo principal do notebook 10.
GRAPH_EXPERIMENTS = [
 {'name':'GCN_completo_ponderado','arch':'GCN','edge':'weighted_abs','pool':'meanmax','features':'fisher'},
 {'name':'GAT_completo_ponderado','arch':'GAT','edge':'weighted_abs','pool':'meanmax','features':'fisher'},
 {'name':'GCN_nao_ponderado','arch':'GCN','edge':'binary_abs','pool':'meanmax','features':'fisher'},
 {'name':'GCN_somente_positivas','arch':'GCN','edge':'positive','pool':'meanmax','features':'fisher'},
 {'name':'GCN_somente_negativas','arch':'GCN','edge':'negative','pool':'meanmax','features':'fisher'},
 {'name':'GCN_pool_atencao','arch':'GCN','edge':'weighted_abs','pool':'attention','features':'fisher'},
 {'name':'GCN_GAT_hibrido','arch':'HYBRID','edge':'weighted_abs','pool':'meanmax','features':'fisher'},
 {'name':'hibrido_sem_GCN','arch':'GAT','edge':'weighted_abs','pool':'meanmax','features':'fisher'},
 {'name':'hibrido_sem_GAT','arch':'GCN','edge':'weighted_abs','pool':'meanmax','features':'fisher'},
]
RUN_ENSEMBLE=True
print('Experimentos de grafo:',len(GRAPH_EXPERIMENTS),'| MODO_RAPIDO:',MODO_RAPIDO)


## Modelos e treinamento

O notebook executa variantes de GCN/GAT e um ensemble selecionado pela validação interna. O modelo BrainGNN e a comparação principal entre GCN/GAT/BrainGNN estão no notebook 10.


In [ ]:
# 4. Matrizes por participante, modelos e utilitários
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N=av.n_rois_de(d['X'].shape[1]); ii,jj=np.triu_indices(N,1)
Z=np.zeros((len(d['y']),N,N),np.float32); Z[:,ii,jj]=d['X']; Z[:,jj,ii]=d['X']
R=np.tanh(Z).astype(np.float32); np.fill_diagonal(R[0],0) # diagonal é zero para todos por construção
class GNN(nn.Module):
 def __init__(self,arch='GCN',pool='meanmax',hid=32,drop=.45):
  super().__init__()
  if arch not in {'GCN','GAT','HYBRID'}: raise ValueError(f'Arquitetura GNN não suportada: {arch}')
  self.arch=arch; self.pool=pool
  self.g1=nn.Linear(N,hid); self.g2=nn.Linear(hid,hid)
  self.a1=nn.Linear(N,hid); self.a2=nn.Linear(hid,hid)
  self.attpool=nn.Linear(hid,1); self.drop=nn.Dropout(drop)
  dim=(hid*2 if arch!='HYBRID' else hid*4) if pool=='meanmax' else (hid if arch!='HYBRID' else hid*2)
  self.head=nn.Sequential(nn.Linear(dim,32),nn.ELU(),nn.Dropout(drop),nn.Linear(32,1))
 def conv(self,H,A,layer,kind):
  if kind=='GCN':
   I=torch.eye(N,device=H.device)[None]; M=A+I; deg=M.sum(-1).clamp_min(1e-6).rsqrt(); return deg[:,:,None]*M*deg[:,None,:] @ layer(H)
  if kind=='GAT':
   q=layer(H); e=F.leaky_relu((q@q.transpose(1,2))/np.sqrt(q.shape[-1]),.2).masked_fill(A<=0,-1e9)
   return torch.softmax(e,-1)@q
  neighbours=torch.bmm(A/(A.sum(-1,keepdim=True).clamp_min(1)),H)
  return layer(torch.cat([H,neighbours],dim=-1))
 def forward(self,X,A):
  branches=[]
  kinds=['GCN','GAT'] if self.arch=='HYBRID' else [self.arch]
  for kind,ls in [('GCN',(self.g1,self.g2)),('GAT',(self.a1,self.a2))]:
   if kind not in kinds: continue
   H=F.elu(self.conv(X,A,ls[0],kind)); H=self.drop(H); H=F.elu(self.conv(H,A,ls[1],kind)); branches.append(H)
  H=torch.cat(branches,-1) if len(branches)>1 else branches[0]
  if self.pool=='attention': pooled=(torch.softmax(self.attpool(H),dim=1)*H).sum(1)
  else: pooled=torch.cat([H.mean(1),H.max(1).values],-1)
  return self.head(self.drop(pooled)).flatten()

def seeded(seed):
 random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
 if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
def adj_for(idx, mode):
 rr=R[idx]
 if mode=='positive': A=np.maximum(rr,0)
 elif mode=='negative': A=np.maximum(-rr,0)
 elif mode=='binary_abs': A=(np.abs(rr)>=np.quantile(np.abs(rr[:,ii,jj]),.8,axis=1)[:,None,None]).astype(np.float32)
 else: A=np.abs(rr)
 A[:,np.arange(N),np.arange(N)]=0
 return torch.tensor(A,dtype=torch.float32,device=DEVICE)
def normalise_mats(train_idx):
 mu=float(Z[train_idx][:,~np.eye(N,dtype=bool)].mean()); sd=float(Z[train_idx][:,~np.eye(N,dtype=bool)].std()+1e-8)
 out=(Z-mu)/sd; out[:,np.arange(N),np.arange(N)]=0; return out.astype(np.float32)
def train_net(model, X, A, y, train, val, epochs=EPOCHS, lr=5e-4, wd=1e-3, l1=0., drop=.4):
 model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=wd)
 wt=torch.tensor(float((y[train]==0).sum()/max(1,(y[train]==1).sum())),device=DEVICE)
 lossfn=nn.BCEWithLogitsLoss(pos_weight=wt); best=-1; state=None; bestep=epochs; wait=0
 yt=torch.tensor(y,dtype=torch.float32,device=DEVICE)
 for ep in range(epochs):
  model.train(); order=np.random.permutation(train)
  for st in range(0,len(order),BATCH):
   b=order[st:st+BATCH]
   if len(b)<2: continue
   opt.zero_grad(set_to_none=True); logits=model(X[b],A[b]); loss=lossfn(logits,yt[b])
   if l1: loss=loss+l1*sum(p.abs().sum() for p in model.parameters())
   loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),2.); opt.step()
  model.eval()
  with torch.no_grad(): pv=torch.sigmoid(model(X[val],A[val])).cpu().numpy() if val is not None else None
  auc=roc_auc_score(y[val],pv) if val is not None and len(np.unique(y[val]))==2 else np.nan
  if val is None: continue
  if auc>best+1e-4: best=auc; state=copy.deepcopy(model.state_dict()); bestep=ep+1; wait=0
  else: wait+=1
  if wait>=PATIENCE: break
 if state is not None: model.load_state_dict(state)
 return model, bestep, best
def probs(model,X,A,idx):
 model.eval(); out=[]
 with torch.no_grad():
  for s in range(0,len(idx),BATCH):
   b=idx[s:s+BATCH]; out.extend(torch.sigmoid(model(X[b],A[b])).cpu().numpy())
 return np.asarray(out,dtype=np.float32)
print('Torch:',torch.__version__,'| device:',DEVICE)


## Execução aninhada, checkpoints e retomada

O split interno é agrupado por centro e escolhido pelo mesmo utilitário do projeto. Candidatos de regularização/learning rate são comparados internamente; melhor época vem do early stopping interno. Para cada variante concluída, salvam-se modelo, probabilidades, limiar interno e parâmetros. A chave de cache inclui hash dos dados, configuração, fold e sementes; mudanças em dados/configuração invalidam automaticamente o checkpoint.

In [ ]:
# 5. Métricas, cache seguro, treinamento externo e escrita progressiva
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
METRICS=['auc_roc','accuracy','balanced_accuracy','precision','recall','specificity','f1']
def calc_metrics(y,p,thr):
 pred=(p>=thr).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
 return dict(auc_roc=roc_auc_score(y,p),accuracy=accuracy_score(y,pred),balanced_accuracy=balanced_accuracy_score(y,pred),precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),specificity=tn/(tn+fp) if tn+fp else np.nan,f1=f1_score(y,pred,zero_division=0))
def threshold(y,p):
 if y is None or len(np.unique(y))<2:return .5
 ts=np.unique(np.round(p,4)); return float(max(ts,key=lambda t:(balanced_accuracy_score(y,p>=t),-abs(t-.5))))
def key_for(name,fold,cfg): return hashlib.sha256(json.dumps([DATA_HASH,name,int(fold),cfg,SEMENTE_BASE],sort_keys=True).encode()).hexdigest()
rows=[]; predrows=[]; artifacts={}
def run_graph(cfg,fold):
 tr=np.where(d['fold_id']!=fold)[0]; te=np.where(d['fold_id']==fold)[0]
 splits,info=av.splits_internos(d['y'][tr],d['sites'][tr],k_max=3,seed=SEMENTE_BASE)
 itr,iva=(tr[splits[0][0]],tr[splits[0][1]]) if info['viavel'] else (tr,None)
 key=key_for(cfg['name'],fold,cfg); cp=CK/f'{key}.pt'; meta=CK/f'{key}.json'
 if cp.exists() and meta.exists():
  saved=torch.load(cp,map_location='cpu',weights_only=False); p=saved['prob']; thr=saved['threshold']; params=saved['params']
 else:
  X=Z.copy(); mu=float(Z[tr][:,~np.eye(N,dtype=bool)].mean()); sd=float(Z[tr][:,~np.eye(N,dtype=bool)].std()+1e-8); X=((X-mu)/sd).astype(np.float32); X[:,np.arange(N),np.arange(N)]=0
  A=adj_for(np.arange(len(d['y'])),cfg['edge'])
  best=None
  # regularização e learning rate são escolhidos só por validação interna
  for lr,wd,l1,drop in [(5e-4,1e-3,0.,.4),(2e-4,1e-2,1e-6,.55)]:
   seeded(SEMENTE_BASE+fold); m=GNN(cfg['arch'],cfg['pool'],drop=drop)
   m,ep,auc=train_net(m,torch.tensor(X,device=DEVICE),A,d['y'],itr,iva,lr=lr,wd=wd,l1=l1)
   score=auc if np.isfinite(auc) else -1
   if best is None or score>best[0]: best=(score,ep,lr,wd,l1,drop)
  _,ep,lr,wd,l1,drop=best
  # refit no treino externo completo durante épocas escolhidas; limiar fica interno
  m=GNN(cfg['arch'],cfg['pool'],drop=drop); m,_ep,_auc=train_net(m,torch.tensor(X,device=DEVICE),A,d['y'],tr,None,epochs=ep,lr=lr,wd=wd,l1=l1)
  # para obter limiar, treina seleção interna escolhida e avalia validação
  seeded(SEMENTE_BASE+fold+700); mv=GNN(cfg['arch'],cfg['pool'],drop=drop); mv,_,_=train_net(mv,torch.tensor(X,device=DEVICE),A,d['y'],itr,iva,epochs=ep,lr=lr,wd=wd,l1=l1)
  pv=probs(mv,torch.tensor(X,device=DEVICE),A,iva) if iva is not None else None; thr=threshold(d['y'][iva],pv) if pv is not None else .5
  p=probs(m,torch.tensor(X,device=DEVICE),A,te)
  params=dict(lr=lr,weight_decay=wd,l1=l1,dropout=drop,epochs=ep,internal_auc=float(best[0]),internal_split=info)
  torch.save({'state':m.state_dict(),'prob':p,'threshold':thr,'params':params},cp); meta.write_text(json.dumps({'data_hash':DATA_HASH,'config':cfg,'fold':int(fold),'params':params},indent=2))
 return te,p,thr,params,key


In [ ]:
def commit(name,fold,result):
 te,p,thr,params,key=result
 met=calc_metrics(d['y'][te],p,thr)
 row=dict(modelo=name,fold=int(fold),n_teste=len(te),threshold=thr,data_hash=DATA_HASH,checkpoint=key,seed=SEMENTE_BASE,**met,**{f'hp_{k}':v for k,v in params.items() if np.isscalar(v)})
 # idempotência pelo par modelo/fold/configuração
 global rows,predrows
 rows=[r for r in rows if not (r['modelo']==name and r['fold']==int(fold))]; rows.append(row)
 predrows=[r for r in predrows if not (r['modelo']==name and r['fold']==int(fold))]
 predrows.extend(dict(modelo=name,fold=int(fold),id=d['ids'][i],site=d['sites'][i],y=int(d['y'][i]),prob=float(p[j]),pred=int(p[j]>=thr),threshold=thr) for j,i in enumerate(te))
 pd.DataFrame(rows).sort_values(['modelo','fold']).to_csv(OUT/'13_metricas_modelo_fold.csv',index=False)
 pd.DataFrame(predrows).sort_values(['modelo','fold','id']).to_csv(OUT/'13_predicoes_oof.csv',index=False)

def run_all():
 global rows,predrows
 # carregar progresso só quando fingerprint coincide
 rp=OUT/'13_metricas_modelo_fold.csv'; pp=OUT/'13_predicoes_oof.csv'
 if rp.exists() and pp.exists():
  old=pd.read_csv(rp); oldp=pd.read_csv(pp)
  if 'data_hash' in old and set(old.data_hash)=={DATA_HASH}:
   rows=old.to_dict('records'); predrows=oldp.to_dict('records')
 for fold in folds:
  for cfg in GRAPH_EXPERIMENTS:
   if not any(r['modelo']==cfg['name'] and int(r['fold'])==int(fold) for r in rows): commit(cfg['name'],fold,run_graph(cfg,fold))
  print('Fold',fold,'salvo progressivamente em',OUT)
 # ensemble: seleciona modelos com AUC interna por fold (não pelo resultado externo), média de probabilidade
 if RUN_ENSEMBLE:
  per=pd.DataFrame(rows); names=[x['name'] for x in GRAPH_EXPERIMENTS]
  for fold in folds:
   te=np.where(d['fold_id']==fold)[0]; tr=np.where(d['fold_id']!=fold)[0]
   avail=per[(per.fold==fold)&per.modelo.isin(names)]
   # ranking via AUC interna armazenada no checkpoint params; fallback apenas modelos elegíveis
   ranked=[]
   for _,r in avail.iterrows():
    if pd.notna(r.get('hp_internal_auc')): ranked.append((float(r['hp_internal_auc']),r['modelo']))
   chosen=[n for _,n in sorted(ranked,reverse=True)[:3]]
   if not chosen: continue
   probs_by=[]
   for n in chosen:
    q=pd.DataFrame(predrows); q=q[(q.modelo==n)&(q.fold==fold)].set_index('id').loc[d['ids'][te]]
    probs_by.append(q.prob.to_numpy())
   pe=np.mean(probs_by,axis=0); thr=.5
   # thresholds use mean inner score only if internal probabilities available; conservative fixed .5
   ename='Ensemble_top3_validacao_interna'; met=calc_metrics(d['y'][te],pe,thr)
   rows=[r for r in rows if not (r['modelo']==ename and r['fold']==int(fold))]
   rows.append(dict(modelo=ename,fold=int(fold),n_teste=len(te),threshold=thr,data_hash=DATA_HASH,checkpoint='ensemble:'+','.join(chosen),seed=SEMENTE_BASE,selection='internal_auc',**met))
   predrows=[r for r in predrows if not (r['modelo']==ename and r['fold']==int(fold))]
   predrows.extend(dict(modelo=ename,fold=int(fold),id=d['ids'][i],site=d['sites'][i],y=int(d['y'][i]),prob=float(pe[j]),pred=int(pe[j]>=thr),threshold=thr) for j,i in enumerate(te))
   pd.DataFrame(rows).to_csv(rp,index=False); pd.DataFrame(predrows).to_csv(pp,index=False)
 return pd.DataFrame(rows),pd.DataFrame(predrows)
# Para iniciar: descomente a linha seguinte e execute. Interrupções são retomáveis.
metrics, oof = run_all()


## Ablação e relatório

A tabela final compara cada variante com a GCN completa ponderada. O ensemble seleciona configurações exclusivamente pela validação interna; as predições externas não participam da seleção.


In [ ]:
# 7. Agregação, ablação, referência histórica, gráficos e relatório
import matplotlib.pyplot as plt

def finalizar(metrics=None,oof=None):
 metrics=pd.read_csv(OUT/'13_metricas_modelo_fold.csv') if metrics is None else metrics
 oof=pd.read_csv(OUT/'13_predicoes_oof.csv') if oof is None else oof
 summary=metrics.groupby('modelo')[METRICS].agg(['mean','std']).round(4)
 summary.to_csv(OUT/'13_resumo_modelos.csv')
 # desempenho agregado diretamente pelas predições OOF concatenadas
 agg=[]
 for name,g in oof.groupby('modelo'):
  met=calc_metrics(g.y.to_numpy(),g.prob.to_numpy(),.5)
  # métricas de classe usam limiar escolhido no treino interno de cada fold
  tn,fp,fn,tp=confusion_matrix(g.y.to_numpy(),g.pred.to_numpy(),labels=[0,1]).ravel()
  met.update(accuracy=accuracy_score(g.y,g.pred),balanced_accuracy=balanced_accuracy_score(g.y,g.pred),precision=precision_score(g.y,g.pred,zero_division=0),recall=recall_score(g.y,g.pred,zero_division=0),specificity=tn/(tn+fp) if tn+fp else np.nan,f1=f1_score(g.y,g.pred,zero_division=0))
  agg.append(dict(modelo=name,**met))
 agg=pd.DataFrame(agg); agg.to_csv(OUT/'13_metricas_oof_agregadas.csv',index=False)
 base='GCN_completo_ponderado'; ref=metrics[metrics.modelo==base].groupby('fold')[METRICS].mean().mean()
 abl=[]
 for name,g in metrics.groupby('modelo'):
  if name==base: continue
  v=g.groupby('fold')[METRICS].mean().mean()
  abl.append(dict(configuracao=name,referencia=base,**{m+'_media_fold':v[m] for m in METRICS},**{'delta_'+m:v[m]-ref[m] for m in METRICS}))
 ab=pd.DataFrame(abl); ab.to_csv(OUT/'13_resultados_ablacao.csv',index=False)
 # referência SVM e resultados de GCN/GAT previamente calculados
 hist=[dict(modelo='SVM-RBF_histórico',auc_roc=.712,origem='referência informada/Results 08-12')]
 for file,model,col in [('10_resumo_por_fold.csv','GCN_anterior',None),('10_resumo_por_fold.csv','GAT_anterior',None)]:
  try:
   df=pd.read_csv(PATHS['results']/file,header=[0,1],index_col=0)
   key={'GCN_anterior':'GCN','GAT_anterior':'GAT'}[model]
   hist.append(dict(modelo=model,auc_roc=float(df.loc[key,('auc_roc','mean')]),origem=file))
  except Exception: pass
 pd.DataFrame(hist).to_csv(OUT/'13_referencias_historicas.csv',index=False)
 # gráfico de AUC OOF e comparativo por fold
 fig,ax=plt.subplots(figsize=(12,5)); ax.bar(agg.modelo,agg.auc_roc,color='#4472C4'); ax.axhline(.712,color='#C00000',ls='--',label='SVM-RBF histórico ~0,712'); ax.set_ylabel('AUC-ROC OOF'); ax.tick_params(axis='x',rotation=75); ax.legend(); fig.tight_layout(); fig.savefig(OUT/'13_auc_comparacao.png',dpi=160); plt.close(fig)
 piv=metrics.pivot(index='fold',columns='modelo',values='auc_roc'); ax=piv.plot(marker='o',figsize=(12,5)); ax.set_ylabel('AUC-ROC por fold'); ax.set_title('Generalização por fold externo'); plt.tight_layout(); plt.savefig(OUT/'13_auc_por_fold.png',dpi=160); plt.close()
 # relatório interpretativo estritamente baseado nos resultados observados
 lines=['# Relatório dos reexperimentos — ABIDE I / CC200','',f'Dataset SHA256: `{DATA_HASH}`. Folds externos por centro: {folds}.','', 'Os valores abaixo vêm das previsões OOF dos experimentos executados. O SVM-RBF (~0,712) é apenas a referência histórica.','']
 if len(agg):
  top=agg.sort_values('auc_roc',ascending=False).iloc[0]
  lines += [f'Maior AUC OOF observada: **{top.modelo} = {top.auc_roc:.3f}**. Isso descreve estes folds; não garante desempenho futuro.','']
 if len(ab):
  lines += ['## Efeito das ablações vs GCN completo', '']
  for _,r in ab.sort_values('delta_auc_roc_media_fold',ascending=False).iterrows():
   lines.append(f"- {r.configuracao}: ΔAUC média por fold {r.delta_auc_roc_media_fold:+.3f}; Δbalanced accuracy {r.delta_balanced_accuracy_media_fold:+.3f}.")
 lines += ['', 'Ver `13_metricas_modelo_fold.csv`, `13_resultados_ablacao.csv`, `13_predicoes_oof.csv` e os gráficos nesta pasta. Variantes sem artefatos extras (fenótipos/séries temporais/topologia) não foram simuladas. A seleção de features funcionais não foi aplicada: a principal usa todas as conexões.','']
 (OUT/'13_relatorio.md').write_text('\n'.join(lines),encoding='utf-8')
 print(summary); print('Artefatos em:',OUT)
 return summary,ab,agg

summary, ablation, oof_summary = finalizar(metrics, oof)


## Observações metodológicas e execução

1. Execute as células 1–7 para carregar/verificar os artefatos e definir os modelos. Para iniciar os treinamentos, execute `metrics, oof = run_all()` na célula 6; após completar, execute `finalizar(metrics, oof)` na última célula.
2. Cada modelo/fold grava checkpoint imediatamente. Se interrompido, retome executando `run_all()`; folds completos com mesmo hash/configuração são ignorados.
3. `13_metricas_modelo_fold.csv` contém métricas por fold. `13_predicoes_oof.csv` contém uma previsão por participante/modelo. `13_resultados_ablacao.csv` contém diferenças em relação à GCN completa. Gráficos e relatório são criados pela função `finalizar`.
4. Para comparação limpa entre componentes, mantenha a mesma semente e folds. As opções fenotípicas e temporais/topológicas só devem ser habilitadas após inspeção e alinhamento por `id`; o modo padrão não inventa nem presume essas variáveis.
6. Resultados são exploratórios, sujeitos a incerteza entre centros e ao tamanho da amostra. AUC OOF e médias de folds devem ser apresentadas juntas; não interpretar pequenas diferenças como evidência conclusiva.